In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_excel("datanew.xlsx")

df_raw.shape
df_raw.info()
df_raw.isna().sum()
df_raw.duplicated().sum()
df_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 12 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Дата                                       104 non-null    int64  
 1   Период                                     104 non-null    int64  
 2   Уровень безработицы, %                     104 non-null    float64
 3   Индекс инвестиций, %                       104 non-null    float64
 4   Номинально начисл. заработная плата, руб.  104 non-null    int64  
 5   Средняя цена 1 кв м, руб.                  104 non-null    float64
 6   Разность безрботицы                        103 non-null    float64
 7   ВРП_млн_руб                                26 non-null     float64
 8   Население_трудоспособное_тыс               24 non-null     float64
 9   Unnamed: 9                                 0 non-null      float64
 10  Unnamed: 10                          

,Дата,Период,"Уровень безработицы, %","Индекс инвестиций, %","Номинально начисл. заработная плата, руб.","Средняя цена 1 кв м, руб.",Разность безрботицы,ВРП_млн_руб,Население_трудоспособное_тыс,Unnamed: 9,Unnamed: 10
count,104.00000,104.000000,104.000000,104.000000,104.000000,104.000000,103.000000,2.600000e+01,24.000000,0.0,0.0
mean,2012.50000,2.500000,5.199423,111.110577,29026.538462,51543.277308,-0.100000,1.025152e+06,2523.458333,NaN,NaN
std,7.53632,1.123448,2.754553,11.960095,23162.503963,52517.040288,0.489277,6.276748e+05,154.731178,NaN,NaN
min,2000.00000,1.000000,1.700000,82.800000,1850.000000,2850.000000,-2.000000,1.685900e+05,2240.000000,NaN,NaN
25%,2006.00000,1.750000,3.300000,103.375000,8650.000000,6820.000000,-0.200000,4.801950e+05,2382.500000,NaN,NaN
50%,2012.50000,2.500000,4.200000,111.500000,23800.000000,44353.220000,0.000000,1.048785e+06,2582.500000,NaN,NaN
75%,2019.00000,3.250000,6.900000,120.200000,44257.500000,65857.162500,0.000000,1.441648e+06,2659.000000,NaN,NaN
max,2025.00000,4.000000,12.100000,130.300000,87500.000000,201000.000000,1.700000,2.250000e+06,2680.000000,NaN,NaN


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path


BASE_DIR = Path("datanew.xlsx").resolve().parent.parent
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "datanew.xlsx"
PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "tatarstan_economic_data_clean.csv"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

df = pd.read_excel(RAW_DATA_PATH)

print("INITIAL DATASET")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

technical_columns = [

    column
    for column in df.columns
    if column.startswith("Unnamed:")
]

df = df.drop(columns=technical_columns)
print("\nRemoved technical columns:")
print(technical_columns)

column_mapping = {

    "Дата": "year",
    "Период": "quarter",
    "Уровень безработицы, %": "unemployment_rate",
    "Индекс инвестиций, %": "investment_index",
    "Номинально начисл. заработная плата, руб.": "average_salary",
    "Средняя цена 1 кв м, руб.": "housing_price",
    "Разность безрботицы": "unemployment_diff",
    "ВРП_млн_руб": "grp_mln_rub",
    "Население_трудоспособное_тыс": "working_age_population_thousands"
}
df = df.rename(columns=column_mapping)


df["year"] = df["year"].astype(int)
df["quarter"] = df["quarter"].astype(int)
numeric_columns = [
    "unemployment_rate",
    "investment_index",
    "average_salary",
    "housing_price",
    "unemployment_diff",
    "grp_mln_rub",
    "working_age_population_thousands"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")
    

df = df.sort_values(by=["year", "quarter"]).reset_index(drop=True)


duplicates = df.duplicated(
    subset=["year", "quarter"]).sum()
print("\nDuplicate year-quarter combinations:", duplicates)

if duplicates > 0:
    df = df.drop_duplicates(
        subset=["year", "quarter"],
        keep="first")


expected_quarters = (df.groupby("year")["quarter"].nunique())

years_with_incomplete_data = expected_quarters[
    expected_quarters != 4
]

print("\nYears with incomplete quarterly data:")
print(years_with_incomplete_data)


df["unemployment_diff"] = (df["unemployment_rate"] .diff())


print("\nMissing values after cleaning:")
print(df.isna().sum())

yearly_columns = [
    "grp_mln_rub",
    "working_age_population_thousands"
]

print("\nYearly variable completeness:")

print(df.groupby("year")[yearly_columns].first().isna().sum())


annual_quarterly_columns = [
    "unemployment_rate",
    "investment_index",
    "average_salary",
    "housing_price"
]

df_annual = (df.groupby("year")[annual_quarterly_columns].mean().reset_index())

annual_yearly_columns = [
    "grp_mln_rub",
    "working_age_population_thousands"
]

df_yearly = (df.groupby("year")[annual_yearly_columns].first().reset_index())

df_annual = df_annual.merge(
    df_yearly,
    on="year",
    how="left"
)


df_annual["unemployment_diff"] = (df_annual["unemployment_rate"].diff())
df_annual["investment_diff"] = (df_annual["investment_index"].diff())

df_annual["grp_log_growth"] = (np.log(df_annual["grp_mln_rub"]).diff())
df_annual["salary_log_growth"] = (np.log(df_annual["average_salary"]).diff())

df_annual["housing_price_log_growth"] = (np.log(df_annual["housing_price"]).diff())


print("FINAL ANNUAL DATASET")
print(f"Rows: {df_annual.shape[0]}")
print(f"Columns: {df_annual.shape[1]}")
print("\nMissing values:")
print(df_annual.isna().sum())
print("\nYear range:")
print(df_annual["year"].min(),
    "-",
    df_annual["year"].max()
)

PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_annual.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nClean annual dataset saved to:")

print(PROCESSED_DATA_PATH)
print("DATA QUALITY REPORT")
print(f"Observations: {len(df_annual)}")
print(f"Years: {df_annual['year'].min()}–{df_annual['year'].max()}")

print("\nMissing values:")
missing_values = df_annual.isna().sum()
print(missing_values[missing_values > 0])

print("\nDuplicate rows:")
print(df_annual.duplicated().sum())

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DATA_PATH = (
    PROCESSED_DATA_DIR / "tatarstan_economic_data_clean.csv"
)

df_annual.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
    encoding="utf-8-sig"
)
print("\nProcessed dataset saved:")
print(PROCESSED_DATA_PATH)


INITIAL DATASET
Rows: 104
Columns: 12

Column types:
Дата                                           int64
Период                                         int64
Уровень безработицы, %                       float64
Индекс инвестиций, %                         float64
Номинально начисл. заработная плата, руб.      int64
Средняя цена 1 кв м, руб.                    float64
Разность безрботицы                          float64
ВРП_млн_руб                                  float64
Население_трудоспособное_тыс                 float64
Unnamed: 9                                   float64
Unnamed: 10                                  float64
Unnamed: 11                                      str
dtype: object

Missing values:
Дата                                           0
Период                                         0
Уровень безработицы, %                         0
Индекс инвестиций, %                           0
Номинально начисл. заработная плата, руб.      0
Средняя цена 1 кв м, руб.          

In [11]:
df_annual[
    [
        "year",
        "unemployment_diff",
        "investment_diff",
        "grp_log_growth",
        "salary_log_growth",
        "housing_price_log_growth"
    ]
]

,year,unemployment_diff,investment_diff,grp_log_growth,salary_log_growth,housing_price_log_growth
0,2000,NaN,NaN,NaN,NaN,NaN
1,2001,-1.300,-6.200,0.141573,-0.164139,0.118952
2,2002,-1.200,-3.200,0.140145,0.654575,0.136642
3,2003,-1.200,4.800,0.158661,0.216379,0.120196
4,2004,-0.900,6.500,0.176596,0.233049,0.164084
5,2005,-0.600,3.600,0.192881,0.229574,0.160267
6,2006,-0.500,-1.400,0.188102,0.208796,0.172400
7,2007,-0.600,-3.500,0.182234,0.225684,0.178158
8,2008,-0.300,-12.800,0.202820,0.203016,0.191482
9,2009,1.700,-26.200,-0.040735,0.075398,-0.065958
